# Drug-Patient Interaction Prediction with Quantum GNN

This notebook demonstrates the complete pipeline for predicting drug-patient interactions using a quantum graph neural network (QGNN).

## Overview

We will:
1. Load drug molecular descriptors from PDB data
2. Generate synthetic patient data
3. Create a bipartite graph of drug-patient interactions
4. Train a quantum GNN model
5. Evaluate and visualize results

## Setup and Imports

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

# Import the drug-patient QGNN package
from drug_patient_qgnn import (
    DrugPatientDataProcessor,
    QuantumDrugPatientGNN,
    DrugPatientTrainer,
    PatientFeatures,
    set_seed,
    print_model_summary,
    print_device_info,
    plot_training_history,
    calculate_metrics,
    print_metrics
)

# Set random seed for reproducibility
set_seed(42)

print("✓ Imports successful!")

## Check Available Compute Devices

In [ ]:
print_device_info()

## Step 1: Load Drug Data from PDB Descriptors

We'll load 3D molecular descriptors from protein-ligand binding data.

In [ ]:
# Initialize data processor
processor = DrugPatientDataProcessor(
    data_dir="/media/priyanshu/SD/othercode/data",
    seed=42
)

# Load drug features from PDB descriptors
print("Loading drug data from PDB descriptors...")
n_drugs = processor.load_protein_ligand_data(max_samples=50)

# If no real data found, create synthetic drugs
if n_drugs == 0:
    print("\nNo PDB data found. Creating synthetic drug data...")
    for i in range(50):
        drug_id = f"synthetic_drug_{i}"
        features = np.random.randn(11)
        processor.graph.add_drug(drug_id, features)
    print(f"Created {processor.graph.num_drugs()} synthetic drugs")

print(f"\n✓ Loaded {processor.graph.num_drugs()} drugs")

## Step 2: Generate Synthetic Patient Data

We'll generate realistic synthetic patient features including:
- Demographics (age, sex, weight)
- Genetic markers
- Comorbidities
- Lab values
- Prior medications

In [ ]:
# Generate synthetic patients
n_patients = 100
processor.create_synthetic_patient_data(n_patients=n_patients)

print(f"✓ Generated {n_patients} synthetic patients")

# Examine a sample patient
sample_patient = list(processor.graph.patient_nodes.values())[0]
print("\nSample Patient:")
print(f"  ID: {sample_patient.patient_id}")
print(f"  Age: {sample_patient.age:.1f}")
print(f"  Sex: {'Male' if sample_patient.sex == 1 else 'Female'}")
print(f"  Weight: {sample_patient.weight:.1f} kg")
print(f"  Genetic markers: {len(sample_patient.genetic_markers)}")
print(f"  Comorbidities: {sample_patient.comorbidities.sum()}/{len(sample_patient.comorbidities)}")

## Step 3: Create Drug-Patient Interactions

Generate synthetic interaction data with realistic efficacy, dose, and outcome distributions.

In [ ]:
# Create interactions (5% of possible drug-patient pairs)
processor.create_synthetic_interactions(interaction_rate=0.05)

# Display dataset statistics
stats = processor.get_statistics()
print("\nDataset Statistics:")
print("=" * 50)
for key, value in stats.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")
print("=" * 50)

## Step 4: Visualize the Bipartite Graph Structure

In [ ]:
# Get graph statistics
graph = processor.graph
edge_index, edge_features = graph.get_edge_index()
labels = graph.get_edge_labels()

# Plot statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Graph structure
axes[0].bar(['Drugs', 'Patients', 'Interactions'], 
           [graph.num_drugs(), graph.num_patients(), graph.num_edges()],
           color=['steelblue', 'coral', 'seagreen'])
axes[0].set_ylabel('Count')
axes[0].set_title('Bipartite Graph Structure')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Outcome distribution
axes[1].pie([np.sum(labels == 0), np.sum(labels == 1)], 
           labels=['Failure', 'Success'],
           autopct='%1.1f%%',
           colors=['lightcoral', 'lightgreen'])
axes[1].set_title('Outcome Distribution')

# Plot 3: Efficacy distribution
efficacy_values = edge_features[:, 0]  # Efficacy is first feature
axes[2].hist(efficacy_values, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Efficacy')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Treatment Efficacy Distribution')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Create Quantum GNN Model

We'll create a quantum GNN with:
- Drug and patient encoders (classical neural networks)
- Quantum interaction layer (variational quantum circuit)
- Classical output head

In [ ]:
# Get feature dimensions
drug_dim = stats['drug_feature_dim']
patient_dim = stats['patient_feature_dim']

# Create model
use_quantum = True  # Set to False for classical baseline

model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=4,       # 4 qubits per side (8 total)
    num_qlayers=2,      # 2 variational layers
    hidden_dim=64,
    use_quantum=use_quantum
)

print_model_summary(model, drug_dim, patient_dim)

if use_quantum:
    print("\n⚛ Using QUANTUM mode")
else:
    print("\n🔷 Using CLASSICAL mode")

## Step 6: Train the Model

In [ ]:
# Create trainer
trainer = DrugPatientTrainer(
    model,
    learning_rate=0.001,
    device=None  # Auto-detect (cuda/mps/cpu)
)

# Train model
history = trainer.fit(
    graph,
    epochs=50,
    batch_size=32,
    val_split=0.2,
    verbose=2
)

## Step 7: Visualize Training Progress

In [ ]:
plot_training_history(history)

## Step 8: Evaluate Model Performance

In [ ]:
# Get all edges for evaluation
n_edges = graph.num_edges()
all_indices = np.arange(n_edges)

# Get predictions
model.eval()
drug_features = torch.tensor(graph.get_drug_features_matrix(), dtype=torch.float32)
patient_features = torch.tensor(graph.get_patient_features_matrix(), dtype=torch.float32)

# Get edge predictions
edge_index, _ = graph.get_edge_index()
drug_indices = edge_index[0]
patient_indices = edge_index[1]

with torch.no_grad():
    edge_drug_features = drug_features[drug_indices].to(trainer.device)
    edge_patient_features = patient_features[patient_indices].to(trainer.device)
    predictions = model(edge_drug_features, edge_patient_features).cpu().numpy().flatten()

# Get true labels
true_labels = graph.get_edge_labels()

# Calculate metrics
pred_labels = (predictions >= 0.5).astype(int)
metrics = calculate_metrics(true_labels, pred_labels, predictions)

print_metrics(metrics, title="Model Performance Metrics")

## Step 9: Visualize Predictions

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Confusion Matrix
cm = confusion_matrix(true_labels, pred_labels)
im = axes[0].imshow(cm, cmap='Blues', aspect='auto')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Predicted Fail', 'Predicted Success'])
axes[0].set_yticklabels(['True Fail', 'True Success'])
axes[0].set_title('Confusion Matrix')

# Add text annotations
for i in range(2):
    for j in range(2):
        text = axes[0].text(j, i, cm[i, j], ha="center", va="center", color="red", fontsize=20)

plt.colorbar(im, ax=axes[0])

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(true_labels, predictions)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

# Plot 3: Prediction Distribution
axes[2].hist(predictions[true_labels == 0], bins=20, alpha=0.5, label='Actual Fail', color='red')
axes[2].hist(predictions[true_labels == 1], bins=20, alpha=0.5, label='Actual Success', color='green')
axes[2].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[2].set_xlabel('Predicted Probability')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Prediction Distribution')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Make Predictions on New Drug-Patient Pairs

In [ ]:
# Select a random drug and patient
drug_idx = 0
patient_idx = 0

# Get their features
drug_features_single = drug_features[drug_idx:drug_idx+1].to(trainer.device)
patient_features_single = patient_features[patient_idx:patient_idx+1].to(trainer.device)

# Make prediction
model.eval()
with torch.no_grad():
    outcome_prob = model(drug_features_single, patient_features_single).item()

print("\nPrediction for New Drug-Patient Pair:")
print("=" * 50)
print(f"Drug ID:     {list(graph.drug_nodes.keys())[drug_idx]}")
print(f"Patient ID:  {list(graph.patient_nodes.keys())[patient_idx]}")
print(f"\nSuccess Probability: {outcome_prob:.2%}")
print(f"Prediction:          {'SUCCESS' if outcome_prob >= 0.5 else 'FAILURE'}")
print("=" * 50)

## Step 11: Save Model and Results

In [ ]:
# Save model
model_path = "trained_qgnn_model.pt"
torch.save(model.state_dict(), model_path)
print(f"✓ Model saved to: {model_path}")

# Save graph data
graph_path = "drug_patient_graph.pkl"
processor.save_graph(graph_path)
print(f"✓ Graph saved to: {graph_path}")

# Save checkpoint
checkpoint_path = "training_checkpoint.pt"
trainer.save_checkpoint(checkpoint_path)
print(f"✓ Checkpoint saved to: {checkpoint_path}")

## Step 12: Compare Quantum vs Classical Performance

Let's train a classical baseline for comparison.

In [ ]:
# Create classical model
classical_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=4,
    num_qlayers=2,
    hidden_dim=64,
    use_quantum=False  # Classical mode
)

# Train classical model
classical_trainer = DrugPatientTrainer(
    classical_model,
    learning_rate=0.001
)

print("\nTraining classical baseline...")
classical_history = classical_trainer.fit(
    graph,
    epochs=50,
    batch_size=32,
    val_split=0.2,
    verbose=1
)

In [ ]:
# Compare performance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot validation loss
axes[0].plot(history['val_loss'], label='Quantum GNN', marker='o', markersize=3)
axes[0].plot(classical_history['val_loss'], label='Classical NN', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Validation Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot validation accuracy
axes[1].plot(history['val_acc'], label='Quantum GNN', marker='o', markersize=3)
axes[1].plot(classical_history['val_acc'], label='Classical NN', marker='s', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final comparison
print("\nFinal Performance Comparison:")
print("=" * 60)
print(f"{'Metric':<20s} {'Quantum GNN':>15s} {'Classical NN':>15s}")
print("=" * 60)
print(f"{'Val Loss':<20s} {history['val_loss'][-1]:>15.4f} {classical_history['val_loss'][-1]:>15.4f}")
print(f"{'Val Accuracy':<20s} {history['val_acc'][-1]:>15.4f} {classical_history['val_acc'][-1]:>15.4f}")
print(f"{'Val AUC-ROC':<20s} {history['val_auc'][-1]:>15.4f} {classical_history['val_auc'][-1]:>15.4f}")
print("=" * 60)

## Summary

In this notebook, we:

1. ✓ Loaded drug molecular descriptors from PDB data
2. ✓ Generated synthetic patient clinical data
3. ✓ Created a bipartite graph structure for drug-patient interactions
4. ✓ Built and trained a Quantum GNN model
5. ✓ Evaluated model performance with multiple metrics
6. ✓ Compared quantum vs classical approaches
7. ✓ Made predictions on new drug-patient pairs

## Next Steps

- Load real patient data from EHR systems
- Experiment with different quantum circuit architectures
- Add multi-task learning for multiple outcomes
- Implement attention mechanisms for interpretability
- Scale up to larger datasets and more qubits